# RANDOM FOREST

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, median_absolute_error, mean_absolute_percentage_error

# All of our preprocessing helper functions are in this notebook
%run 05_0_preproc_helpers.ipynb  

# this is the target column - the value we want to predict 
TARGET_COL = "price"  

# separate features and target variable from the full training datase
y = full_train_dataset[TARGET_COL].copy()
X = full_train_dataset.drop(columns=[TARGET_COL]).copy()

# lists previously defined in 05_0_preproc_helpers.ipynb
numeric_features = num_feat                      # ['year', 'mileage', ...]
categorical_features = cat_feat                  # ['Brand', 'model', 'transmission', 'fuelType']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Num features:", numeric_features)
print("Cat features:", categorical_features)


X shape: (75973, 11)
y shape: (75973,)
Num features: ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'paintQuality%', 'previousOwners']
Cat features: ['Brand', 'model', 'transmission', 'fuelType']


__Some notes:__

The following preprocessing steps have already been applied to full_train_dataset (in the 05_0_preproc_helpers notebook):
- The columns carID and hasDamage were removed.
- The functions fill_unknown, basic_string_transformer and correct_invalid_brands_in_df were applied across the dataset.
- As a result, the categorical columns no longer contain missing values (all of the previously missing entries are marked as "UNKNOWN"), and all categorical values have been normalized: uppercase, accent-free, strictly alphanumeric and without spaces. Brand values that partially matched a valid brand name have also been corrected.
- Numerical columns may still contain missing values and categorical features still require additional processing, which will be handled using a proper fit/transform logic during cross-validation to avoid data leakage.

## 2. Randomized Hyperparameter Search with K-Fold Cross-Validation

This section defines and executes a full randomized hyperparameter search for a Random Forest regressor using an 8-fold K-Fold cross-validation strategy. A search space of candidate hyperparameters is sampled (n_iter = 50), and for each sampled configuration the following steps are performed inside every fold:

Train/validation split according to the KFold indices.

Numerical preprocessing using a strict fit/transform logic: median-based imputations, domain-aware imputers for year, mileage, engineSize, tax, mpg, paintQuality%, and previousOwners.

Categorical cleaning and harmonization, resolving invalid or ambiguous brands, models, transmissions, and fuel types, based only on the training split.

Categorical encoding: target encoding for high-cardinality features (Brand, model) and one-hot encoding for the remaining categorical features.

Numerical scaling using StandardScaler (fit on train, transform on validation).

Feature assembly by concatenating normalized categorical features with the scaled numerical features.

Model training and evaluation with the current Random Forest hyperparameters, computing RMSE, MAE, and R² for both the training and validation sets.

Metrics from all folds are aggregated to produce mean RMSE, MAE, and a combined score (0.5·RMSE + 0.5·MAE). The system keeps track of the best-performing configurations under each criterion. All results are logged to a file and stored in a summary dataframe for inspection.

In [2]:
# set core configuration for cross validation
N_SPLITS = 8 # number of folds used in k-fold cross validation
RANDOM_STATE = 42

# creation of the KFold object that will generate the train/validation splits
kf = KFold(
    n_splits=N_SPLITS, # number of folds, in this case will be 8 folds
    shuffle=True,  # shuffle data before splitting to make folds more balanced
    random_state=RANDOM_STATE # using the fixed seed 
)

# ------ // ------
# Definition of the hyperparameters that will be tested in the Randomized Search

param_distributions = {
    # number of trees in the forest: 
    "n_estimators": [200, 400, 600, 800, 1000], 
    # maximum depth of each tree:
    "max_depth": [5, 7, 9, 11, 15, 20],
    # minimum number of samples required to split an internal node:
    "min_samples_split": [2, 4, 6, 10],
    # minimum number of samples required to be at a leaf node:
    "min_samples_leaf": [1, 2, 4, 8],
    # number of features to consider when looking for the best split:
    "max_features": [None], # using all features
    "bootstrap": [True, False], # whether bootstrap samples are used when building trees
}

N_RANDOM_CONFIGS = 50 # number of random different hyperparameter configurations to try


# this sampler will generate random combinations of hyperparameters
# from the previously defined search space 
sampler = ParameterSampler(
    param_distributions, # the dictionary with the hyperparameter search space
    n_iter=N_RANDOM_CONFIGS, # number of different random configurations to generate
    random_state=RANDOM_STATE 
)

search_results = [] # list that will store summary results for each hyperparameter configuration

# best configuration according to validation rmse
best_rmse = np.inf # our initial best rmse is infinite so any real rmse will be smaller
best_config = None # will store the params corresponding to the best rmse

# best configuration according to validation mae
best_mae = np.inf
best_config_mae = None

# best configuration according to a combined metric 0.5 * rmse + 0.5 * mae on validation
best_combo = np.inf
best_config_combo = None

# we have a log file to store detailed logs of the random search process
# this is useful for later analysis of the results, to avoid losing information if the notebook crashes
# and following the progress of the search
log_path = "rf_random_search_log.txt"

# opens the log file for writing
with open(log_path, "w", encoding="utf-8") as log_file:

    # this is a helper function to write messages to both console and log file
    def log(msg: str):
        print(msg) # print to stdout so progress is visible during execution
        log_file.write(msg + "\n") # also write the same message to the log file
        log_file.flush() # flush so log is updated immediately on disk

    log("# =============================")
    log("# INÍCIO DO RANDOM SEARCH RF")
    log("# =============================")
    log(f"N_SPLITS = {N_SPLITS}, N_RANDOM_CONFIGS = {N_RANDOM_CONFIGS}")
    log(f"param_distributions = {param_distributions}")

    # -----------------------------
    # main loop of the random search with cross validation
    # each iteration uses one sampled hyperparameter configuration
    # -----------------------------
    for config_id, params in enumerate(sampler, start=1):
        log("")
        log(f"######## CONFIG {config_id}/{N_RANDOM_CONFIGS} ########")
        log(f"Parâmetros: {params}")

        # lists to store validation metrics per fold for this configuration
        fold_rmses = []
        fold_maes  = []
        fold_r2s   = []

        # lists to store training metrics per fold for this configuration
        fold_rmses_train = []
        fold_maes_train  = []
        fold_r2s_train   = []

        # cross validation loop for the current hyperparameter configuration
        for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):

            log("")
            log(f"[CONFIG {config_id}] ==== FOLD {fold}/{N_SPLITS} ====")

            # ---------------------------------
            # 1) split the data into train and validation for the current fold
            # ---------------------------------
            X_train = X.iloc[train_idx].copy() # training features for the current fold
            X_val   = X.iloc[val_idx].copy() # validation features for the current fold
            y_train = y.iloc[train_idx].copy() # training targets for the current fold
            y_val   = y.iloc[val_idx].copy() # validation targets for the current fold

            log(f"[C{config_id}|F{fold}] X_train shape: {X_train.shape}, X_val shape: {X_val.shape}")
            log(f"[C{config_id}|F{fold}] y_train shape: {y_train.shape}, y_val shape: {y_val.shape}")

            # log missing values BEFORE imputations
            # --> for the numeric features
            log(f"[C{config_id}|F{fold}] NaNs ANTES imputação numérica (apenas numéricas):")
            log(str(X_train[numeric_features].isna().sum()))
            # --> for the categorical features (we are not supposed to have NaNs here because they were already temporarily handled as UNKNOWN, but just in case)
            log(f"[C{config_id}|F{fold}] NaNs ANTES (categóricas):")
            log(str(X_train[categorical_features].isna().sum()))
            # count how many "UNKNOWN" exist in the categorical features before further processing
            unknown_counts_before = (X_train[categorical_features] == "UNKNOWN").sum()
            log(f"[C{config_id}|F{fold}] 'UNKNOWN' ANTES (categóricas):")
            log(str(unknown_counts_before))

            # ---------------------------------
            # 2) numerical preprocessing 
            #    using fit on train and transform on val
            #    each "state" object stores statistics or rules learned from train
            # ---------------------------------

            # 2.1) YEAR 
            # impute year based on median year per model learned from training data
            year_state = fit_year_median(
                X_train,
                year_col="year",
                model_col="model"
            )
            X_train = transform_year_with_model_median(X_train, state=year_state)
            X_val   = transform_year_with_model_median(X_val,   state=year_state)

            # 2.2) MILEAGE
            # impute mileage using a custom imputer that can also enforce absolute values
            mileage_state = fit_mileage_imputer(
                X_train,
                mileage_col="mileage",
                do_abs=True
            )
            X_train = transform_mileage_imputer(X_train, state=mileage_state)
            X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

            # 2.3) ENGINE SIZE
            # impute engine size using a imputer fitted on the training data
            engine_state = fit_engine_size_imputer(
                X_train,
                engine_col="engineSize"
            )
            X_train = transform_engine_size_imputer(X_train, state=engine_state)
            X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

            # 2.4) TAX
            # impute tax with a custom imputer 
            tax_state = fit_tax_imputer(
                X_train,
                tax_col="tax",
                do_abs=True
            )
            X_train = transform_tax_imputer(X_train, state=tax_state)
            X_val   = transform_tax_imputer(X_val,   state=tax_state)

            # 2.5) MPG
            mpg_state = fit_mpg_imputer(
                X_train,
                mpg_col="mpg",
                do_abs=True
            )
            X_train = transform_mpg_imputer(X_train, state=mpg_state)
            X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

            # 2.6) PAINT QUALITY
            paint_state = fit_paint_quality_imputer(
                X_train,
                paint_col="paintQuality%"
            )
            X_train = transform_paint_quality_imputer(X_train, state=paint_state)
            X_val   = transform_paint_quality_imputer(X_val,   state=paint_state)

            # 2.7) PREVIOUS OWNERS
            owners_state = fit_previous_owners_imputer(
                X_train,
                owners_col="previousOwners",
                year_col="year",
                mileage_col="mileage"
            )
            X_train = transform_previous_owners_imputer(X_train, state=owners_state)
            X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

            # --
            # log some statistics after numerical imputations
            # to check if they were, generally, preprocessed correctly
            log(f"[C{config_id}|F{fold}] Após imputação numérica: X_train shape = {X_train.shape}, X_val shape = {X_val.shape}")
            log(f"[C{config_id}|F{fold}] NaNs DEPOIS imputação numérica (apenas numéricas):")
            log(str(X_train[numeric_features].isna().sum()))
            # --

            # ---------------------------------
            # 2.8) BRAND 
            # ---------------------------------
            brand_state = fit_ambiguous_brand_resolver(
                train_df=X_train,
                valid_brands=valid_brands,
                brand_col="Brand",
                model_col="model",
                year_col="year",
            )
            X_train, brand_corr_train, brand_still_invalid_train = transform_ambiguous_brands(
                X_train,
                brand_state,
            )
            X_val, brand_corr_val, brand_still_invalid_val = transform_ambiguous_brands(
                X_val,
                brand_state,
            )
            # --
            log(f"[C{config_id}|F{fold}] Após resolver Brand:")
            log("  Nº brands ainda inválidas (train): " + str(len(brand_still_invalid_train)))
            log("  Exemplo brands inválidas (train): " + str(brand_still_invalid_train[:10]))
            # --

            # ---------------------------------
            # 2.9) MODEL
            # ---------------------------------
            model_state = fit_invalid_model_resolver(
                train_df=X_train,
                valid_models_by_brand=valid_models_by_brand,
                brand_col="Brand",
                model_col="model",
                year_col="year",
                fuel_col="fuelType",
                mpg_col="mpg",
            )
            X_train, model_corr_train, model_still_invalid_train = transform_invalid_models(
                X_train,
                model_state,
            )
            X_val, model_corr_val, model_still_invalid_val = transform_invalid_models(
                X_val,
                model_state,
            )
            # --
            log(f"[C{config_id}|F{fold}] Após resolver model:")
            log("  Nº models ainda inválidos (train): " + str(len(model_still_invalid_train)))
            log("  Exemplo models inválidos (train): " + str(model_still_invalid_train[:10]))
            # -- 

            # ---------------------------------
            # 2.10) TRANSMISSION
            # ---------------------------------
            transm_state = fit_transmission_resolver(
                train_df=X_train,
                valid_transmissions=valid_transmissions,
                transm_col="transmission",
                brand_col="Brand",
                model_col="model",
                fuel_col="fuelType",
            )
            X_train, transm_corr_train, transm_still_invalid_train = transform_transmission_resolver(
                X_train,
                transm_state,
            )
            X_val, transm_corr_val, transm_still_invalid_val = transform_transmission_resolver(
                X_val,
                transm_state,
            )
            # --
            log(f"[C{config_id}|F{fold}] Após resolver transmission:")
            log("  Valores distintos (train): " +
                str(sorted(X_train["transmission"].astype(str).str.upper().unique())))
            log("  Ainda problemáticos (train): " + str(transm_still_invalid_train[:10]))
            # --
            
            # ---------------------------------
            # 2.11) FUEL TYPE
            # ---------------------------------
            fuel_state = fit_fueltype_resolver(
                train_df=X_train,
                valid_fueltypes=valid_fueltypes,
                fuel_col="fuelType",
                brand_col="Brand",
                model_col="model",
                transm_col="transmission",
            )
            X_train, fuel_corr_train, fuel_still_invalid_train = transform_fueltype_resolver(
                X_train,
                fuel_state,
            )
            X_val, fuel_corr_val, fuel_still_invalid_val = transform_fueltype_resolver(
                X_val,
                fuel_state,
            )
            # --
            log(f"[C{config_id}|F{fold}] Após resolver fuelType:")
            log("  Valores distintos (train): " +
                str(sorted(X_train["fuelType"].astype(str).str.upper().unique())))
            log("  Ainda problemáticos (train): " + str(fuel_still_invalid_train[:10]))
            # --

            # ---------------------------------
            # Some extra logs for checking if the preprocessing went well
            # ---------------------------------
            log(f"[C{config_id}|F{fold}] --- RESUMO NUMÉRICAS (train, após imputação) ---")
            num_means_train = X_train[numeric_features].mean()
            num_nans_train  = X_train[numeric_features].isna().sum()
            log("  Médias numéricas (train):")
            log(str(num_means_train))
            log("  NaNs numéricas (train):")
            log(str(num_nans_train))

            num_means_val = X_val[numeric_features].mean()
            num_nans_val  = X_val[numeric_features].isna().sum()
            log("  Médias numéricas (val):")
            log(str(num_means_val))
            log("  NaNs numéricas (val):")
            log(str(num_nans_val))

            log(f"[C{config_id}|F{fold}] --- RESUMO CATEGÓRICAS (train, após resolver tudo) ---")
            cat_nans_train     = X_train[categorical_features].isna().sum()
            cat_unknown_train  = (X_train[categorical_features] == "UNKNOWN").sum()
            log("  NaNs por categórica (train):")
            log(str(cat_nans_train))
            log("  'UNKNOWN' por categórica (train):")
            log(str(cat_unknown_train))

            log(f"[C{config_id}|F{fold}] Valores distintos por categórica (train):")
            for col in categorical_features:
                uniques = sorted(X_train[col].dropna().astype(str).unique())
                if len(uniques) > 30:
                    shown = uniques[:30]
                    log(f"    {col}: {len(uniques)} valores distintos. Primeiros 30: {shown} ...")
                else:
                    log(f"    {col}: {len(uniques)} valores distintos: {uniques}")

            # left UNKNOWN 
            unknown_mask_train = (X_train[categorical_features] == "UNKNOWN")
            unknown_mask_val   = (X_val[categorical_features]   == "UNKNOWN")

            unknown_counts_train = unknown_mask_train.sum()
            unknown_counts_val   = unknown_mask_val.sum()

            rows_with_unknown_train = unknown_mask_train.any(axis=1).sum()
            rows_with_unknown_val   = unknown_mask_val.any(axis=1).sum()

            log(f"[C{config_id}|F{fold}] 'UNKNOWN' DEPOIS resolver (categóricas, train):")
            log(str(unknown_counts_train))
            log(f"[C{config_id}|F{fold}] 'UNKNOWN' DEPOIS resolver (categóricas, val):")
            log(str(unknown_counts_val))
            log(f"[C{config_id}|F{fold}] Nº de linhas com >=1 UNKNOWN (train): {rows_with_unknown_train}")
            log(f"[C{config_id}|F{fold}] Nº de linhas com >=1 UNKNOWN (val)  : {rows_with_unknown_val}")


            # ---------------------------------
            # 3) categorical encoding
            #    (Target encoding for Brand+model, One-Hot for the rest)
            # ---------------------------------
            high_card_features = ["Brand", "model"]  # features with high cardinality => treatment with target encoding
            low_card_features  = [c for c in categorical_features if c not in high_card_features]

            # --
            log(f"[C{config_id}|F{fold}] high_card_features = {high_card_features}")
            log(f"[C{config_id}|F{fold}] low_card_features  = {low_card_features}")
            # --

            # 3.1) Target Encoding para Brand e model
            te = MyTargetEncoder(smoothing=5)
            te.fit(X_train[high_card_features], y_train)

            X_train_high = te.transform(X_train[high_card_features])
            X_val_high   = te.transform(X_val[high_card_features])

            # 3.2) One-Hot Encoding para as restantes categóricas
            ohe = MyOneHotEncoder()
            ohe.fit(X_train[low_card_features])

            X_train_low = ohe.transform(X_train[low_card_features])
            X_val_low   = ohe.transform(X_val[low_card_features])

            X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
            X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

            log(f"[C{config_id}|F{fold}] X_train_cat shape: {X_train_cat.shape}")
            log(f"[C{config_id}|F{fold}] X_val_cat   shape: {X_val_cat.shape}")

            log(f"[C{config_id}|F{fold}] NaNs DEPOIS encoding (X_train_cat, total): {X_train_cat.isna().sum().sum()}")
            log(f"[C{config_id}|F{fold}] NaNs DEPOIS encoding (X_val_cat, total): {X_val_cat.isna().sum().sum()}")

            unknown_cols_train = [col for col in X_train_cat.columns if "UNKNOWN" in str(col)]
            unknown_cols_val   = [col for col in X_val_cat.columns if "UNKNOWN" in str(col)]

            log(f"[C{config_id}|F{fold}] Colunas encodadas com 'UNKNOWN' (train): {unknown_cols_train}")
            log(f"[C{config_id}|F{fold}] Colunas encodadas com 'UNKNOWN' (val)  : {unknown_cols_val}")

            # ---------------------------------
            # 4) NUMERICAL SCALING
            # ---------------------------------
            scaler = StandardScaler()
            X_train_num = scaler.fit_transform(X_train[numeric_features]) # fit on train and transform train
            X_val_num   = scaler.transform(X_val[numeric_features]) # transform validation using same scaler

            # convert numpy arrays back to dataframes with meaningful column names and consistent indices
            X_train_num_df = pd.DataFrame(
                X_train_num,
                index=X_train.index,
                columns=[f"num_{col}" for col in numeric_features]
            )
            X_val_num_df = pd.DataFrame(
                X_val_num,
                index=X_val.index,
                columns=[f"num_{col}" for col in numeric_features]
            )

            log(f"[C{config_id}|F{fold}] X_train_num_df shape: {X_train_num_df.shape}")
            log(f"[C{config_id}|F{fold}] X_val_num_df   shape: {X_val_num_df.shape}")

            # ---------------------------------
            # 5) concatenate scaled numeric and encoded categorical features
            # ---------------------------------
            X_train_final = pd.concat(
                [X_train_num_df, X_train_cat],
                axis=1
            )
            X_val_final = pd.concat(
                [X_val_num_df, X_val_cat],
                axis=1
            )

            log(f"[C{config_id}|F{fold}] X_train_final shape: {X_train_final.shape}")
            log(f"[C{config_id}|F{fold}] X_val_final   shape: {X_val_final.shape}")

            # ---------------------------------
            # 6) instantiate and train the random forest regressor for this configuration
            # ---------------------------------
            rf = RandomForestRegressor(
                random_state=RANDOM_STATE,
                n_jobs=-1, # uses all available cores 
                **params # current hyperparameter configuration
            )

            log(f"[C{config_id}|F{fold}] A treinar RandomForestRegressor...")
            rf.fit(X_train_final, y_train) # train the model on the fully processed training data

            # generate predictions on training and validation sets for metric computation
            y_pred_train = rf.predict(X_train_final)
            y_pred_val   = rf.predict(X_val_final)

            # ---------------------------------
            # 7) compute metrics per fold for both train and validation
            #    used to monitor performance and detect potential overfitting/underfitting
            # ---------------------------------
            # Validation metrics:
            mse_val  = mean_squared_error(y_val, y_pred_val)
            rmse_val = np.sqrt(mse_val)
            mae_val  = mean_absolute_error(y_val, y_pred_val)
            r2_val   = r2_score(y_val, y_pred_val)

            # TRAIN metrics: 
            mse_tr  = mean_squared_error(y_train, y_pred_train)
            rmse_tr = np.sqrt(mse_tr)
            mae_tr  = mean_absolute_error(y_train, y_pred_train)
            r2_tr   = r2_score(y_train, y_pred_train)

            log(f"[C{config_id}|F{fold}] TRAIN -> RMSE: {rmse_tr:.4f} | MAE: {mae_tr:.4f} | R²: {r2_tr:.4f}")
            log(f"[C{config_id}|F{fold}] VAL   -> RMSE: {rmse_val:.4f} | MAE: {mae_val:.4f} | R²: {r2_val:.4f}")

            # store the validation metrics for this fold
            fold_rmses.append(rmse_val)
            fold_maes.append(mae_val)
            fold_r2s.append(r2_val)

            fold_rmses_train.append(rmse_tr)
            fold_maes_train.append(mae_tr)
            fold_r2s_train.append(r2_tr)

        # average metrics of all folds for this configuration
        mean_rmse_val = np.mean(fold_rmses)
        mean_mae_val  = np.mean(fold_maes)
        mean_r2_val   = np.mean(fold_r2s)

        mean_rmse_tr = np.mean(fold_rmses_train)
        mean_mae_tr  = np.mean(fold_maes_train)
        mean_r2_tr   = np.mean(fold_r2s_train)

        # combined score giving equal weight to rmse and mae on validation
        combo_score = 0.5 * mean_rmse_val + 0.5 * mean_mae_val

        log("")
        log(f"Config {config_id} - TRAIN RMSE médio: {mean_rmse_tr:.4f} | MAE: {mean_mae_tr:.4f} | R²: {mean_r2_tr:.4f}")
        log(f"Config {config_id} - VAL   RMSE médio: {mean_rmse_val:.4f} | MAE: {mean_mae_val:.4f} | R²: {mean_r2_val:.4f}")
        log(f"Config {config_id} - Score combinado (0.5*RMSE + 0.5*MAE) [VAL]: {combo_score:.4f}")


        # store a record with configuration, metrics and combined score to the search results list
        search_results.append({
            "config_id": config_id,
            **params,
            "rmse_train_mean": mean_rmse_tr,
            "mae_train_mean": mean_mae_tr,
            "r2_train_mean": mean_r2_tr,
            "rmse_mean": mean_rmse_val,
            "mae_mean": mean_mae_val,
            "r2_mean": mean_r2_val,
            "combo_score": combo_score,
        })

        # update (or not) RMSE (VAL)
        if mean_rmse_val < best_rmse:
            best_rmse = mean_rmse_val
            best_config = {**params}
            log(f"[NOVO MELHOR RMSE] Config {config_id} com RMSE médio (VAL) = {best_rmse:.4f}")

        # update (or not) MAE (VAL)
        if mean_mae_val < best_mae:
            best_mae = mean_mae_val
            best_config_mae = {**params}
            log(f"[NOVO MELHOR MAE] Config {config_id} com MAE médio (VAL) = {best_mae:.4f}")

        # update (or not) best combined score (VAL)
        if combo_score < best_combo:
            best_combo = combo_score
            best_config_combo = {**params}
            log(f"[NOVO MELHOR COMBINADO] Config {config_id} com score = {best_combo:.4f}")


    # after iterating over all sampled configurations, log a summary of the best ones
    log("")
    log("")
    log("# =============================")
    log("# FIM DO RANDOM SEARCH RF")
    log("# =============================")
    log(f"Melhor configuração (min RMSE VAL): {best_config}")
    log(f"Melhor RMSE médio (VAL): {best_rmse:.4f}")
    log(f"Melhor configuração (min MAE VAL): {best_config_mae}")
    log(f"Melhor MAE médio (VAL): {best_mae:.4f}")
    log(f"Melhor configuração (score combinado VAL): {best_config_combo}")
    log(f"Melhor score combinado (VAL): {best_combo:.4f}")

# -----------------------------
# final summary of the random search (as a dataframe)
# -----------------------------
results_df = pd.DataFrame(search_results) # convert list of dicts to dataframe
results_df_sorted = results_df.sort_values(by="rmse_mean", ascending=True) # sort by validation rmse

display(results_df_sorted.head(10)) # configurations with lowest validation rmse

# print the best configurations according to each criterion
print("\nMelhor configuração encontrada (min RMSE VAL):")
print(best_config)
print("Melhor RMSE médio (VAL):", best_rmse)

print("\nMelhor configuração encontrada (min MAE VAL):")
print(best_config_mae)
print("Melhor MAE médio (VAL):", best_mae)

print("\nMelhor configuração encontrada (score combinado VAL = 0.5*RMSE + 0.5*MAE):")
print(best_config_combo)
print("Melhor score combinado (VAL):", best_combo)

print(f"\nLogs detalhados guardados em: {log_path}")


# =============================
# INÍCIO DO RANDOM SEARCH RF
# =============================
N_SPLITS = 8, N_RANDOM_CONFIGS = 50
param_distributions = {'n_estimators': [200, 400, 600, 800, 1000], 'max_depth': [5, 7, 9, 11, 15, 20], 'min_samples_split': [2, 4, 6, 10], 'min_samples_leaf': [1, 2, 4, 8], 'max_features': [None], 'bootstrap': [True, False]}

######## CONFIG 1/50 ########
Parâmetros: {'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': None, 'max_depth': 15, 'bootstrap': False}

[CONFIG 1] ==== FOLD 1/8 ====
[C1|F1] X_train shape: (66476, 11), X_val shape: (9497, 11)
[C1|F1] y_train shape: (66476,), y_val shape: (9497,)
[C1|F1] NaNs ANTES imputação numérica (apenas numéricas):
year              1306
mileage           1252
engineSize        1344
tax               6940
mpg               6961
paintQuality%     1322
previousOwners    1331
dtype: int64
[C1|F1] NaNs ANTES (categóricas):
Brand           0
model           0
transmission    0
fuelType 

,config_id,n_estimators,min_samples_split,min_samples_leaf,max_features,max_depth,bootstrap,rmse_train_mean,mae_train_mean,r2_train_mean,rmse_mean,mae_mean,r2_mean,combo_score
7,8,1000,6,1,None,15,True,1548.207086,989.858705,0.974716,2281.399465,1346.387853,0.944933,1813.893659
23,24,600,2,2,None,15,True,1570.327201,974.674176,0.973988,2306.579526,1347.249540,0.943704,1826.914533
2,3,200,6,2,None,15,True,1633.937550,1002.769218,0.971837,2317.450564,1350.389879,0.943169,1833.920222
1,2,600,10,8,None,20,True,2050.554099,1134.735515,0.955644,2462.509848,1391.901559,0.935840,1927.205704
17,18,1000,2,8,None,20,True,2050.692388,1134.644669,0.955638,2463.749213,1391.953739,0.935778,1927.851476
5,6,400,2,2,None,11,True,2097.722961,1360.228869,0.953583,2486.367430,1512.866747,0.934650,1999.617088
46,47,200,6,4,None,11,True,2208.045256,1382.650338,0.948572,2529.513894,1519.707825,0.932342,2024.610860
36,37,400,10,8,None,11,True,2353.882558,1421.161918,0.941553,2598.353390,1536.198167,0.928615,2067.275779
33,34,200,4,1,None,9,True,2465.908681,1617.516281,0.935860,2707.709431,1698.046447,0.922572,2202.877939
38,39,800,4,1,None,9,True,2466.303726,1617.307703,0.935840,2708.383249,1698.088314,0.922533,2203.235781



Melhor configuração encontrada (min RMSE VAL):
{'n_estimators': 1000, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 15, 'bootstrap': True}
Melhor RMSE médio (VAL): 2281.3994648627117

Melhor configuração encontrada (min MAE VAL):
{'n_estimators': 1000, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 15, 'bootstrap': True}
Melhor MAE médio (VAL): 1346.387852917274

Melhor configuração encontrada (score combinado VAL = 0.5*RMSE + 0.5*MAE):
{'n_estimators': 1000, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 15, 'bootstrap': True}
Melhor score combinado (VAL): 1813.8936588899928

Logs detalhados guardados em: rf_random_search_log.txt


## 3. Final Model Training and Kaggle Submission Generation

In [ ]:
# ----------------------------------------------------
# 0) final configuration chosen manually based on previous random search results (we used the best RMSE one), 
# but all of the obtained configurations are present in 'rf_random_search_log.txt' notebook for further analysis if needed
#    this dictionary stores the hyperparameters that will be used to train the final model on the full training data
# ----------------------------------------------------
""" 
# this is the fs one, and n_estimators was increased to 1000 for fs_1000
final_config = {
    "n_estimators": 200,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": 0.5,
    "max_depth": 20,
    "bootstrap": True,
    "n_features_to_select": 0.9
}
Note: version rf_final_submission_config_fs is the version 11
and rf_final_submission_config_fs_1000est is the 12 
"""

final_config = {
    "n_estimators": 200,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": 0.5,
    "max_depth": 20,
    "bootstrap": True,
    "n_features_to_select": 0.9
}


n_features_pct = final_config.pop("n_features_to_select", 1.0)
print("Config final usada para treino:", final_config)

# ----------------------------------------------------
# 1) create full training copies of features and target
#    we work on copies of X and y to avoid side effects on any previous objects
# ----------------------------------------------------
X_full = X.copy()
y_full = y.copy()


# list of categorical columns for which we want to enforce string normalization
cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]

# apply deterministic string normalization to the selected columns
# this ensures consistent formatting before fitting imputers and resolvers
for col in cols_to_normalize:
    if col in X_full.columns:
        X_full[col] = fill_unknown(X_full[col])

        X_full = column_string_transformer(
            X_full,
            column=col,
            remove_middle_spaces=True,
            allow_extra_chars=""
        )

invalids = sorted(
    [b for b in X_full['Brand'].unique() if b not in valid_brands],
    key=len
)

# Aplicar a função
X_full, corrections, remaining_invalids = correct_invalid_brands_in_df(
    X_full,
    col='Brand',
    valid_brands=valid_brands,
    invalids=invalids
)

# define high cardinality and low cardinality categorical features as done before
# high cardinality will be target encoded and the remaining categoricals will be one hot encoded (same as before)
high_card_features = ["Brand", "model"]  # features to Target Encoding
low_card_features  = [c for c in categorical_features if c not in high_card_features]

# ----------------------------------------------------
# 2) numerical preprocessing on full training data
#    all fit operations are performed on X_full so that the learned states can be reused on the test set
#    this replicates the same pipeline used inside cross validation but now applied on the entire training data
# ----------------------------------------------------

# 2.1) YEAR
year_state = fit_year_median(
    X_full,
    year_col="year",
    model_col="model"
)
X_full = transform_year_with_model_median(X_full, state=year_state)

# 2.2) MILEAGE
mileage_state = fit_mileage_imputer(
    X_full,
    mileage_col="mileage",
    do_abs=True
)
X_full = transform_mileage_imputer(X_full, state=mileage_state)

# 2.3) ENGINE SIZE
engine_state = fit_engine_size_imputer(
    X_full,
    engine_col="engineSize"
)
X_full = transform_engine_size_imputer(X_full, state=engine_state)

# 2.4) TAX
tax_state = fit_tax_imputer(
    X_full,
    tax_col="tax",
    do_abs=True
)
X_full = transform_tax_imputer(X_full, state=tax_state)

# 2.5) MPG
mpg_state = fit_mpg_imputer(
    X_full,
    mpg_col="mpg",
    do_abs=True
)
X_full = transform_mpg_imputer(X_full, state=mpg_state)

# 2.6) PAINT QUALITY
paint_state = fit_paint_quality_imputer(
    X_full,
    paint_col="paintQuality%"
)
X_full = transform_paint_quality_imputer(X_full, state=paint_state)

# 2.7) PREVIOUS OWNERS
owners_state = fit_previous_owners_imputer(
    X_full,
    owners_col="previousOwners",
    year_col="year",
    mileage_col="mileage"
)
X_full = transform_previous_owners_imputer(X_full, state=owners_state)


# 2.8) BRANDS
brand_state = fit_ambiguous_brand_resolver(
    train_df=X_full,
    valid_brands=valid_brands,
    brand_col="Brand",
    model_col="model",
    year_col="year",
)

X_full, brand_corr_full, brand_still_invalid_full = transform_ambiguous_brands(
    X_full,
    brand_state,
)
print("Após resolver Brand (train full): nº brands ainda inválidas:",
      len(brand_still_invalid_full))

# 2.9) MODELS
model_state = fit_invalid_model_resolver(
    train_df=X_full,
    valid_models_by_brand=valid_models_by_brand,
    brand_col="Brand",
    model_col="model",
    year_col="year",
    fuel_col="fuelType",
    mpg_col="mpg",
)

X_full, model_corr_full, model_still_invalid_full = transform_invalid_models(
    X_full,
    model_state,
)
print("Após resolver model (train full): nº models ainda inválidos:",
      len(model_still_invalid_full))

# 2.10) TRANSMISSION 
transm_state = fit_transmission_resolver(
    train_df=X_full,
    valid_transmissions=valid_transmissions,
    transm_col="transmission",
    brand_col="Brand",
    model_col="model",
    fuel_col="fuelType",
)

X_full, transm_corr_full, transm_still_invalid_full = transform_transmission_resolver(
    X_full,
    transm_state,
)
print("Após resolver transmission (train full): valores distintos:",
      sorted(X_full["transmission"].astype(str).str.upper().unique()))

# 2.11) FUEL TYPE 
fuel_state = fit_fueltype_resolver(
    train_df=X_full,
    valid_fueltypes=valid_fueltypes,
    fuel_col="fuelType",
    brand_col="Brand",
    model_col="model",
    transm_col="transmission",
)

X_full, fuel_corr_full, fuel_still_invalid_full = transform_fueltype_resolver(
    X_full,
    fuel_state,
)
print("Após resolver fuelType (train full): valores distintos:",
      sorted(X_full["fuelType"].astype(str).str.upper().unique()))

# ----------------------------------------------------
# categorical encoding on full training data
#    high cardinality features brand and model are target encoded
#    remaining categorical features are one hot encoded
# ----------------------------------------------------

# fit target encoder on high cardinality features using the full training target
te = MyTargetEncoder(smoothing=5)
te.fit(X_full[high_card_features], y_full)

X_full_high = te.transform(X_full[high_card_features]) # transform brand and model into continuous encoded features

# fit one hot encoder on the remaining categorical features
ohe = MyOneHotEncoder()
ohe.fit(X_full[low_card_features])

X_full_low = ohe.transform(X_full[low_card_features])

X_full_cat = pd.concat([X_full_high, X_full_low], axis=1) # combine target encoded and one hot encoded categorical features into a single dataframe

print("X_full_cat shape:", X_full_cat.shape)

# ----------------------------------------------------
# 5) numerical scaling on full training data
#    standardize numerical features so that they have zero mean and unit variance
# ----------------------------------------------------
scaler = StandardScaler()
X_full_num = scaler.fit_transform(X_full[numeric_features])

# convert back to dataframe with consistent index and prefixed column names
X_full_num_df = pd.DataFrame(
    X_full_num,
    index=X_full.index,
    columns=[f"num_{col}" for col in numeric_features]
)

print("X_full_num_df shape:", X_full_num_df.shape)


# ----------------------------------------------------
# build final training matrix by concatenating scaled numeric and encoded categorical features
#    will be used to train the final model
# ----------------------------------------------------
X_full_final = pd.concat( [X_full_num_df, X_full_cat], axis=1)

print("X_full_final shape:", X_full_final.shape)


selector = None # Inicializa variavel

if n_features_pct < 1.0 and n_features_pct is not None:
    total_cols = X_full_final.shape[1]
    n_feats_final = int(total_cols * n_features_pct)
    n_feats_final = max(1, n_feats_final)
    
    print(f"A selecionar as melhores {n_feats_final} features...")
    
    # Instancia o seletor (usamos RF rápido para selecionar)
    selector = MyRandomForestSelector(
        n_features=n_feats_final, 
        n_estimators=50, 
        random_state=RANDOM_STATE
    )
    
    # Fit no X_full
    selector.fit(X_full_final, y_full)
    
    # Transform (Reduz colunas)
    X_full_final = selector.transform(X_full_final)
    
    print("Features selecionadas:", selector.selected_features_)
else:
    print("Feature selection ignorado (usando todas as features).")

print("X_full_final shape (DEPOIS seleção):", X_full_final.shape)

# ============================================================
# ----------------------------------------------------
# train final random forest model on full training data using the chosen configuration
#    this is the model that will be used to generate predictions for the test set
# ----------------------------------------------------
rf_final = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **final_config
)

rf_final.fit(X_full_final, y_full)
print("Modelo final treinado com sucesso.")

# ----------------------------------------------------
# prepare test features using the exact same pipeline as for training
#    here we only apply transform operations using the states learned on the training data
# ----------------------------------------------------
test_df = pd.read_csv("../../project_data/test.csv")

# normalize string representation for the same set of categorical columns in the test data
cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]

for col in cols_to_normalize:
    if col in test_df.columns:
        # 1. Preencher vazios com UNKNOWN (igual ao treino)
        test_df[col] = fill_unknown(test_df[col])
        
        # 2. Normalizar string
        test_df = column_string_transformer(
            test_df,
            column=col,
            remove_middle_spaces=True,
            allow_extra_chars=""
        )

# select the same base set of numeric and categorical columns as in training
X_test = test_df[numeric_features + categorical_features].copy()

# apply numerical imputers to the test data using training states only
X_test = transform_year_with_model_median(X_test, state=year_state)
X_test = transform_mileage_imputer(X_test, state=mileage_state)
X_test = transform_engine_size_imputer(X_test, state=engine_state)
X_test = transform_tax_imputer(X_test, state=tax_state)
X_test = transform_mpg_imputer(X_test, state=mpg_state)
X_test = transform_paint_quality_imputer(X_test, state=paint_state)
X_test = transform_previous_owners_imputer(X_test, state=owners_state)

# resolve categorical inconsistencies on test set using the same states learned from training
X_test, _, _ = transform_ambiguous_brands(X_test, brand_state)
X_test, _, _ = transform_invalid_models(X_test, model_state)
X_test, _, _ = transform_transmission_resolver(X_test, transm_state)
X_test, _, _ = transform_fueltype_resolver(X_test, fuel_state)

# apply the fitted target encoder and one hot encoder to the test data
X_test_high = te.transform(X_test[high_card_features])
X_test_low  = ohe.transform(X_test[low_card_features])

X_test_cat = pd.concat([X_test_high, X_test_low], axis=1)

# scale numerical test features using the scaler fitted on the full training data
X_test_num = scaler.transform(X_test[numeric_features])

X_test_num_df = pd.DataFrame(
    X_test_num,
    index=X_test.index,
    columns=[f"num_{col}" for col in numeric_features]
)

# build final test matrix by concatenating scaled numeric and encoded categorical test features
X_test_final = pd.concat(
    [X_test_num_df, X_test_cat],
    axis=1
)

if selector is not None:
    # Usa o selector que já foi treinado (fitted) no X_full
    X_test_final = selector.transform(X_test_final)
    print("Seleção aplicada ao teste.")

print("X_test_final shape (DEPOIS seleção):", X_test_final.shape)
# ----------------------------------------------------
# generate test set predictions and build submission file
#    predictions are first inspected as floats and then optionally rounded to integers
# ----------------------------------------------------
y_test_pred = rf_final.predict(X_test_final)

print("Primeiras 10 previsões (float):", y_test_pred[:10])
print("Resumo das previsões (float):")
print(pd.Series(y_test_pred).describe())

# prices round the predictions to the nearest integer
y_test_pred_round = np.round(y_test_pred).astype(int)

# carid and predicted price
submission = pd.DataFrame({
    "carID": test_df["carID"].astype(int),
    "price": y_test_pred
})

submission_path = "rf_final_submission_config_fs.csv"
submission.to_csv(submission_path, index=False)

print("Ficheiro de submissão criado em:", submission_path)


Config final usada para treino: {'n_estimators': 1000, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 15, 'bootstrap': True}
Após resolver Brand (train full): nº brands ainda inválidas: 0
Após resolver model (train full): nº models ainda inválidos: 1
Após resolver transmission (train full): valores distintos: ['AUTOMATIC', 'MANUAL', 'SEMIAUTO']
Após resolver fuelType (train full): valores distintos: ['DIESEL', 'ELECTRIC', 'HYBRID', 'OTHER', 'PETROL']
X_full_cat shape: (75973, 10)
X_full_num_df shape: (75973, 7)
X_full_final shape: (75973, 17)
A selecionar as melhores 15 features...
Features selecionadas: ['model', 'num_mileage', 'num_engineSize', 'num_year', 'num_mpg', 'num_paintQuality%', 'Brand', 'num_tax', 'transmission_MANUAL', 'num_previousOwners', 'fuelType_PETROL', 'fuelType_DIESEL', 'transmission_SEMIAUTO', 'transmission_AUTOMATIC', 'fuelType_HYBRID']
X_full_final shape (DEPOIS seleção): (75973, 15)
Modelo final treinado com sucesso.
Seleção a

## WITH GRID SERACH AND FEATURE SELECTION

In [16]:
# ---------------------------------------------------------
# CORE CONFIGURATION
# ---------------------------------------------------------
N_SPLITS = 8 
RANDOM_STATE = 42

kf = KFold(
    n_splits=N_SPLITS, 
    shuffle=True,  
    random_state=RANDOM_STATE 
)

# ---------------------------------------------------------
# HYPERPARAMETERS
# ---------------------------------------------------------
param_distributions = {
    "n_estimators": [200, 400, 600, 800, 1000], 
    "max_depth": [7, 9, 11, 15, 20],
    "min_samples_split": [2, 4, 6, 10],
    "min_samples_leaf": [1, 2, 4, 6, 8],
    "max_features": ["sqrt", 0.33, 0.5, 1.0], 
    "bootstrap": [True], 
    "n_features_to_select": [1.0, 0.9, 0.8, 0.6, 0.4] 
}

# ---------------------------------------------------------
# RANDOM SEARCH CONFIG
# ---------------------------------------------------------
N_RANDOM_CONFIGS = 250 

sampler = ParameterSampler(
    param_distributions, 
    n_iter=N_RANDOM_CONFIGS, 
    random_state=RANDOM_STATE 
)

search_results = [] 

best_rmse = np.inf 
best_config = None 

best_mae = np.inf
best_config_mae = None

best_combo = np.inf
best_config_combo = None

log_path = "rf_250_random_search_log.txt"

with open(log_path, "w", encoding="utf-8") as log_file:

    def log(msg: str):
        print(msg) 
        log_file.write(msg + "\n") 
        log_file.flush() 

    log("# =============================")
    log(f"# START OF RF RANDOM SEARCH (DETAILED LOGS)")
    log(f"# {N_RANDOM_CONFIGS} ITERATIONS")
    log("# =============================")
    
    # -----------------------------
    # MAIN LOOP
    # -----------------------------
    for config_id, params in enumerate(sampler, start=1):
        log("")
        log(f"######## CONFIG {config_id}/{N_RANDOM_CONFIGS} ########")
        log(f"Parameters: {params}")

        # Separate params
        frac_feats = params.get("n_features_to_select", 1.0)
        rf_params = {k: v for k, v in params.items() if k != "n_features_to_select"}

        # Metrics Lists
        fold_rmses, fold_maes, fold_r2s = [], [], []
        fold_rmses_train, fold_maes_train, fold_r2s_train = [], [], []
        fold_med_ae, fold_mape, fold_bias = [], [], []

        for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
            
            # --- 1) Split Data ---
            X_train = X.iloc[train_idx].copy()
            X_val   = X.iloc[val_idx].copy()
            y_train = y.iloc[train_idx].copy()
            y_val   = y.iloc[val_idx].copy()

            log(f"\n[C{config_id}|F{fold}] Processing fold...")
            
            # --- LOG: Initial State ---
            log(f"  > Initial NaNs (Train Numeric): {X_train[numeric_features].isna().sum().sum()}")
            log(f"  > Initial NaNs (Train Categorical): {X_train[categorical_features].isna().sum().sum()}")
            unknowns_init = (X_train[categorical_features] == "UNKNOWN").sum()
            log(f"  > Initial 'UNKNOWN' counts:\n{unknowns_init[unknowns_init > 0]}")

            # --- 2) Numerical Preprocessing ---
            # 2.1 Year
            year_state = fit_year_median(X_train, year_col="year", model_col="model")
            X_train = transform_year_with_model_median(X_train, state=year_state)
            X_val   = transform_year_with_model_median(X_val,   state=year_state)

            # 2.2 Mileage
            mileage_state = fit_mileage_imputer(X_train, mileage_col="mileage", do_abs=True)
            X_train = transform_mileage_imputer(X_train, state=mileage_state)
            X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

            # 2.3 Engine Size
            engine_state = fit_engine_size_imputer(X_train, engine_col="engineSize")
            X_train = transform_engine_size_imputer(X_train, state=engine_state)
            X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

            # 2.4 Tax
            tax_state = fit_tax_imputer(X_train, tax_col="tax", do_abs=True)
            X_train = transform_tax_imputer(X_train, state=tax_state)
            X_val   = transform_tax_imputer(X_val,   state=tax_state)

            # 2.5 MPG
            mpg_state = fit_mpg_imputer(X_train, mpg_col="mpg", do_abs=True)
            X_train = transform_mpg_imputer(X_train, state=mpg_state)
            X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

            # 2.6 Paint Quality
            paint_state = fit_paint_quality_imputer(X_train, paint_col="paintQuality%")
            X_train = transform_paint_quality_imputer(X_train, state=paint_state)
            X_val   = transform_paint_quality_imputer(X_val,   state=paint_state)

            # 2.7 Previous Owners
            owners_state = fit_previous_owners_imputer(
                X_train, owners_col="previousOwners", year_col="year", mileage_col="mileage"
            )
            X_train = transform_previous_owners_imputer(X_train, state=owners_state)
            X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

            # --- Categorical Resolvers ---
            # 2.8 Brand
            brand_state = fit_ambiguous_brand_resolver(
                X_train, valid_brands=valid_brands, brand_col="Brand", model_col="model", year_col="year"
            )
            X_train, _, _ = transform_ambiguous_brands(X_train, brand_state)
            X_val, _, _   = transform_ambiguous_brands(X_val, brand_state)

            # 2.9 Model
            model_state = fit_invalid_model_resolver(
                X_train, valid_models_by_brand=valid_models_by_brand, 
                brand_col="Brand", model_col="model", year_col="year", fuel_col="fuelType", mpg_col="mpg"
            )
            X_train, _, _ = transform_invalid_models(X_train, model_state)
            X_val, _, _   = transform_invalid_models(X_val, model_state)

            # 2.10 Transmission
            transm_state = fit_transmission_resolver(
                X_train, valid_transmissions=valid_transmissions, transm_col="transmission"
            )
            X_train, _, _ = transform_transmission_resolver(X_train, transm_state)
            X_val, _, _   = transform_transmission_resolver(X_val, transm_state)
            
            # 2.11 Fuel Type
            fuel_state = fit_fueltype_resolver(
                X_train, valid_fueltypes=valid_fueltypes, fuel_col="fuelType"
            )
            X_train, _, _ = transform_fueltype_resolver(X_train, fuel_state)
            X_val, _, _   = transform_fueltype_resolver(X_val, fuel_state)

            # --- LOG: Post-Processing State ---
            log(f"  > Post-Proc NaNs (Train Numeric): {X_train[numeric_features].isna().sum().sum()}")
            unknowns_post = (X_train[categorical_features] == "UNKNOWN").sum()
            log(f"  > Post-Proc 'UNKNOWN' counts:\n{unknowns_post[unknowns_post > 0]}")

            # --- 3) Categorical Encoding ---
            high_card_features = ["Brand", "model"]
            low_card_features  = [c for c in categorical_features if c not in high_card_features]

            te = MyTargetEncoder(smoothing=5)
            te.fit(X_train[high_card_features], y_train)
            X_train_high = te.transform(X_train[high_card_features])
            X_val_high   = te.transform(X_val[high_card_features])

            ohe = MyOneHotEncoder()
            ohe.fit(X_train[low_card_features])
            X_train_low = ohe.transform(X_train[low_card_features])
            X_val_low   = ohe.transform(X_val[low_card_features])

            X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
            X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

            # --- 4) Numerical Scaling ---
            scaler = StandardScaler()
            X_train_num = scaler.fit_transform(X_train[numeric_features])
            X_val_num   = scaler.transform(X_val[numeric_features])

            X_train_num_df = pd.DataFrame(X_train_num, index=X_train.index, columns=[f"num_{col}" for col in numeric_features])
            X_val_num_df = pd.DataFrame(X_val_num, index=X_val.index, columns=[f"num_{col}" for col in numeric_features])

            # --- 5) Concatenate Features ---
            X_train_final = pd.concat([X_train_num_df, X_train_cat], axis=1)
            X_val_final   = pd.concat([X_val_num_df, X_val_cat],   axis=1)
            
            # ============================================================
            # FEATURE SELECTION (Percentage Logic)
            # ============================================================
            total_cols = X_train_final.shape[1]
            if frac_feats >= 1.0:
                n_feats_final = total_cols
            else:
                n_feats_final = int(total_cols * frac_feats)
                n_feats_final = max(1, n_feats_final)

            if n_feats_final < total_cols:
                log(f"  > Feature Selection: Keeping top {n_feats_final} columns ({frac_feats*100:.0f}%). Dropping {total_cols - n_feats_final} columns.")
                selector = MyRandomForestSelector(n_features=n_feats_final, n_estimators=50, random_state=RANDOM_STATE)
                selector.fit(X_train_final, y_train)
                
                dropped_cols = [c for c in X_train_final.columns if c not in selector.selected_features_]
                # Log dropped columns (limited to first 10 to avoid spam)
                log(f"  > Dropped Columns (First 10): {dropped_cols[:10]} ...")
                
                X_train_final = selector.transform(X_train_final)
                X_val_final   = selector.transform(X_val_final)
            else:
                log(f"  > Feature Selection: Skipped (Using all {total_cols} features).")
            # ============================================================

            # --- 6) Train Model ---
            rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **rf_params)
            rf.fit(X_train_final, y_train)

            y_pred_train = rf.predict(X_train_final)
            y_pred_val   = rf.predict(X_val_final)

            # --- 7) METRICS ---
            mse_val, mae_val = mean_squared_error(y_val, y_pred_val), mean_absolute_error(y_val, y_pred_val)
            rmse_val = np.sqrt(mse_val)
            r2_val = r2_score(y_val, y_pred_val)

            mse_tr, mae_tr = mean_squared_error(y_train, y_pred_train), mean_absolute_error(y_train, y_pred_train)
            rmse_tr = np.sqrt(mse_tr)
            r2_tr = r2_score(y_train, y_pred_train)
            
            med_ae_val = median_absolute_error(y_val, y_pred_val)
            mape_val = mean_absolute_percentage_error(y_val, y_pred_val) * 100 
            bias_val = np.mean(y_val - y_pred_val)

            log(f"  > Fold Results: [TRAIN] RMSE={rmse_tr:.2f}, MAE={mae_tr:.2f} | [VAL] RMSE={rmse_val:.2f}, MAE={mae_val:.2f}")

            fold_rmses.append(rmse_val)
            fold_maes.append(mae_val)
            fold_r2s.append(r2_val)
            fold_rmses_train.append(rmse_tr)
            fold_maes_train.append(mae_tr)
            fold_r2s_train.append(r2_tr)
            fold_med_ae.append(med_ae_val)
            fold_mape.append(mape_val)
            fold_bias.append(bias_val)

        # Average metrics
        mean_rmse_val = np.mean(fold_rmses)
        mean_mae_val  = np.mean(fold_maes)
        mean_r2_val   = np.mean(fold_r2s)
        mean_rmse_tr  = np.mean(fold_rmses_train)
        mean_mae_tr   = np.mean(fold_maes_train)
        mean_r2_tr    = np.mean(fold_r2s_train)
        mean_med_ae_val = np.mean(fold_med_ae)
        mean_mape_val   = np.mean(fold_mape)
        mean_bias_val   = np.mean(fold_bias)

        combo_score = 0.5 * mean_rmse_val + 0.5 * mean_mae_val

        log("")
        log(f"Config {config_id} - [VAL AVG] RMSE: {mean_rmse_val:.4f} | MAE: {mean_mae_val:.4f} | MedAE: {mean_med_ae_val:.4f} | Bias: {mean_bias_val:.4f}")
        log(f"Config {config_id} - [TRAIN AVG] RMSE: {mean_rmse_tr:.4f} | MAE: {mean_mae_tr:.4f}")

        search_results.append({
            "config_id": config_id,
            **params,
            "rmse_train_mean": mean_rmse_tr,
            "mae_train_mean": mean_mae_tr,
            "r2_train_mean": mean_r2_tr,
            "rmse_mean": mean_rmse_val,
            "mae_mean": mean_mae_val,
            "r2_mean": mean_r2_val,
            "med_ae_mean": mean_med_ae_val,
            "mape_mean": mean_mape_val,
            "bias_mean": mean_bias_val,
            "combo_score": combo_score,
        })

        if mean_rmse_val < best_rmse:
            best_rmse = mean_rmse_val
            best_config = {**params}
            log(f"[NEW BEST RMSE] Config {config_id}")

        if mean_mae_val < best_mae:
            best_mae = mean_mae_val
            best_config_mae = {**params}
            log(f"[NEW BEST MAE] Config {config_id}")
            
        if combo_score < best_combo:
            best_combo = combo_score
            best_config_combo = {**params}
            log(f"[NEW BEST COMBINED SCORE] Config {config_id}")

    log("# END OF RANDOM SEARCH RF")

# -----------------------------
# Results
# -----------------------------
results_df = pd.DataFrame(search_results)
results_df_sorted = results_df.sort_values(by="rmse_mean", ascending=True)

cols_to_display = ["config_id", "n_estimators", "max_depth", "n_features_to_select", "rmse_mean", "mae_mean", "med_ae_mean", "mape_mean", "bias_mean"]
cols_to_display = [c for c in cols_to_display if c in results_df_sorted.columns]

print("\nTop 10 Configurations by RMSE:")
display(results_df_sorted[cols_to_display].head(10))

print("\n" + "="*50)
print("FINAL RESULTS ANALYSIS")
print("="*50)

# 1. Best RMSE
print("\n--- BEST CONFIGURATION BY RMSE (Root Mean Squared Error) ---")
best_row_rmse = results_df_sorted.iloc[0]
print(f"Config ID: {best_row_rmse['config_id']}")
print(f"Params: {best_config}")
print(f"RMSE: {best_row_rmse['rmse_mean']:.4f}")

# 2. Best MAE
print("\n--- BEST CONFIGURATION BY MAE (Mean Absolute Error) ---")
best_row_mae = results_df.sort_values(by="mae_mean", ascending=True).iloc[0]
print(f"Config ID: {best_row_mae['config_id']}")
print(f"Params: {best_config_mae}")
print(f"MAE: {best_row_mae['mae_mean']:.4f}")

# 3. Best Median Absolute Error
print("\n--- BEST CONFIGURATION BY MEDIAN AE (Robustness Check) ---")
best_row_med = results_df.sort_values(by="med_ae_mean", ascending=True).iloc[0]
best_params_med = {k: best_row_med[k] for k in param_distributions.keys() if k in best_row_med}
print(f"Config ID: {best_row_med['config_id']}")
print(f"Params: {best_params_med}")
print(f"Median AE: {best_row_med['med_ae_mean']:.4f}")

print(f"\nDetailed logs saved at: {log_path}")

# =============================
# START OF RF RANDOM SEARCH (DETAILED LOGS)
# 250 ITERATIONS
# =============================

######## CONFIG 1/250 ########
Parameters: {'n_features_to_select': 0.8, 'n_estimators': 200, 'min_samples_split': 6, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 15, 'bootstrap': True}

[C1|F1] Processing fold...
  > Initial NaNs (Train Numeric): 20456
  > Initial NaNs (Train Categorical): 0
  > Initial 'UNKNOWN' counts:
Brand           1328
model           1314
transmission    1944
fuelType        1296
dtype: Int64
  > Post-Proc NaNs (Train Numeric): 0
  > Post-Proc 'UNKNOWN' counts:
Series([], dtype: int64)
  > Feature Selection: Keeping top 13 columns (80%). Dropping 4 columns.
  > Dropped Columns (First 10): ['transmission_AUTOMATIC', 'fuelType_ELECTRIC', 'fuelType_HYBRID', 'fuelType_OTHER'] ...
  > Fold Results: [TRAIN] RMSE=2113.68, MAE=1247.43 | [VAL] RMSE=2265.38, MAE=1433.38

[C1|F2] Processing fold...
  > Initial NaNs (Train Numeric): 2

,config_id,n_estimators,max_depth,n_features_to_select,rmse_mean,mae_mean,med_ae_mean,mape_mean,bias_mean
88,89,200,20,0.9,2186.695353,1283.851822,808.170233,8.119555,-9.208409
18,19,200,15,0.9,2252.288174,1332.312032,852.154184,8.425247,-1.985789
108,109,1000,20,0.6,2253.914894,1306.799506,816.594399,8.218594,-2.841305
168,169,1000,20,1.0,2262.911298,1317.007693,832.903596,8.459663,-1.084473
142,143,800,15,0.8,2265.962852,1347.482110,868.638947,8.622752,0.531803
162,163,400,15,0.8,2273.985072,1346.756403,846.905252,8.430676,-5.020219
172,173,400,15,0.6,2277.242314,1343.864809,853.439974,8.455649,-3.938415
226,227,200,20,1.0,2278.304672,1323.524003,836.435158,8.483004,-1.758605
57,58,400,20,1.0,2287.602066,1317.764486,817.678036,8.241558,-4.373663
78,79,200,20,1.0,2288.566513,1324.138134,838.042976,8.462163,-2.491834



FINAL RESULTS ANALYSIS

--- BEST CONFIGURATION BY RMSE (Root Mean Squared Error) ---
Config ID: 89
Params: {'n_features_to_select': 0.9, 'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 20, 'bootstrap': True}
RMSE: 2186.6954

--- BEST CONFIGURATION BY MAE (Mean Absolute Error) ---
Config ID: 89
Params: {'n_features_to_select': 0.9, 'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.5, 'max_depth': 20, 'bootstrap': True}
MAE: 1283.8518

--- BEST CONFIGURATION BY MEDIAN AE (Robustness Check) ---
Config ID: 89
Params: {'n_estimators': np.int64(200), 'max_depth': np.int64(20), 'min_samples_split': np.int64(2), 'min_samples_leaf': np.int64(1), 'max_features': 0.5, 'bootstrap': np.True_, 'n_features_to_select': np.float64(0.9)}
Median AE: 808.1702

Detailed logs saved at: rf_250_random_search_log.txt


## TEST 

In [17]:
# ----------------------------------------------------
# 0) final configuration chosen manually based on previous random search results (we used the best RMSE one), 
# but all of the obtained configurations are present in 'rf_random_search_log.txt' notebook for further analysis if needed
#    this dictionary stores the hyperparameters that will be used to train the final model on the full training data
# ----------------------------------------------------
final_config = {
    "n_estimators": 1000,
    "min_samples_split": 6,
    "min_samples_leaf": 1,
    "max_features": None,
    "max_depth": 15,
    "bootstrap": True,
}

n_features_pct = final_config.pop("n_features_to_select", 1.0) # Remove do dict para não dar erro no RF
print(f"Config final RF: {final_config}")
print(f"Features a manter: {n_features_pct * 100}%")

# ----------------------------------------------------
# 1) create full training copies of features and target
#    we work on copies of X and y to avoid side effects on any previous objects
# ----------------------------------------------------
X_full = X.copy()
y_full = y.copy()


# list of categorical columns for which we want to enforce string normalization
cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]

# apply deterministic string normalization to the selected columns
# this ensures consistent formatting before fitting imputers and resolvers
for col in cols_to_normalize:
    if col in X_full.columns:
        X_full[col] = fill_unknown(X_full[col])

        X_full = column_string_transformer(
            X_full,
            column=col,
            remove_middle_spaces=True,
            allow_extra_chars=""
        )

invalids = sorted(
    [b for b in X_full['Brand'].unique() if b not in valid_brands],
    key=len
)

# Aplicar a função
X_full, corrections, remaining_invalids = correct_invalid_brands_in_df(
    X_full,
    col='Brand',
    valid_brands=valid_brands,
    invalids=invalids
)

print(f"Correções aplicadas: {len(corrections)}")
print(f"Brands ainda inválidas (serão tratadas pelo resolver robusto): {len(remaining_invalids)}")


# define high cardinality and low cardinality categorical features as done before
# high cardinality will be target encoded and the remaining categoricals will be one hot encoded (same as before)
high_card_features = ["Brand", "model"]  # features to Target Encoding
low_card_features  = [c for c in categorical_features if c not in high_card_features]

# ----------------------------------------------------
# 2) numerical preprocessing on full training data
#    all fit operations are performed on X_full so that the learned states can be reused on the test set
#    this replicates the same pipeline used inside cross validation but now applied on the entire training data
# ----------------------------------------------------

# 2.1) YEAR
year_state = fit_year_median(
    X_full,
    year_col="year",
    model_col="model"
)
X_full = transform_year_with_model_median(X_full, state=year_state)

# 2.2) MILEAGE
mileage_state = fit_mileage_imputer(
    X_full,
    mileage_col="mileage",
    do_abs=True
)
X_full = transform_mileage_imputer(X_full, state=mileage_state)

# 2.3) ENGINE SIZE
engine_state = fit_engine_size_imputer(
    X_full,
    engine_col="engineSize"
)
X_full = transform_engine_size_imputer(X_full, state=engine_state)

# 2.4) TAX
tax_state = fit_tax_imputer(
    X_full,
    tax_col="tax",
    do_abs=True
)
X_full = transform_tax_imputer(X_full, state=tax_state)

# 2.5) MPG
mpg_state = fit_mpg_imputer(
    X_full,
    mpg_col="mpg",
    do_abs=True
)
X_full = transform_mpg_imputer(X_full, state=mpg_state)

# 2.6) PAINT QUALITY
paint_state = fit_paint_quality_imputer(
    X_full,
    paint_col="paintQuality%"
)
X_full = transform_paint_quality_imputer(X_full, state=paint_state)

# 2.7) PREVIOUS OWNERS
owners_state = fit_previous_owners_imputer(
    X_full,
    owners_col="previousOwners",
    year_col="year",
    mileage_col="mileage"
)
X_full = transform_previous_owners_imputer(X_full, state=owners_state)


# 2.8) BRANDS
brand_state = fit_ambiguous_brand_resolver(
    train_df=X_full,
    valid_brands=valid_brands,
    brand_col="Brand",
    model_col="model",
    year_col="year",
)

X_full, brand_corr_full, brand_still_invalid_full = transform_ambiguous_brands(
    X_full,
    brand_state,
)
print("Após resolver Brand (train full): nº brands ainda inválidas:",
      len(brand_still_invalid_full))

# 2.9) MODELS
model_state = fit_invalid_model_resolver(
    train_df=X_full,
    valid_models_by_brand=valid_models_by_brand,
    brand_col="Brand",
    model_col="model",
    year_col="year",
    fuel_col="fuelType",
    mpg_col="mpg",
)

X_full, model_corr_full, model_still_invalid_full = transform_invalid_models(
    X_full,
    model_state,
)
print("Após resolver model (train full): nº models ainda inválidos:",
      len(model_still_invalid_full))

# 2.10) TRANSMISSION 
transm_state = fit_transmission_resolver(
    train_df=X_full,
    valid_transmissions=valid_transmissions,
    transm_col="transmission",
    brand_col="Brand",
    model_col="model",
    fuel_col="fuelType",
)

X_full, transm_corr_full, transm_still_invalid_full = transform_transmission_resolver(
    X_full,
    transm_state,
)
print("Após resolver transmission (train full): valores distintos:",
      sorted(X_full["transmission"].astype(str).str.upper().unique()))

# 2.11) FUEL TYPE 
fuel_state = fit_fueltype_resolver(
    train_df=X_full,
    valid_fueltypes=valid_fueltypes,
    fuel_col="fuelType",
    brand_col="Brand",
    model_col="model",
    transm_col="transmission",
)

X_full, fuel_corr_full, fuel_still_invalid_full = transform_fueltype_resolver(
    X_full,
    fuel_state,
)
print("Após resolver fuelType (train full): valores distintos:",
      sorted(X_full["fuelType"].astype(str).str.upper().unique()))

# ----------------------------------------------------
# categorical encoding on full training data
#    high cardinality features brand and model are target encoded
#    remaining categorical features are one hot encoded
# ----------------------------------------------------

# fit target encoder on high cardinality features using the full training target
te = MyTargetEncoder(smoothing=5)
te.fit(X_full[high_card_features], y_full)

X_full_high = te.transform(X_full[high_card_features]) # transform brand and model into continuous encoded features

# fit one hot encoder on the remaining categorical features
ohe = MyOneHotEncoder()
ohe.fit(X_full[low_card_features])

X_full_low = ohe.transform(X_full[low_card_features])

X_full_cat = pd.concat([X_full_high, X_full_low], axis=1) # combine target encoded and one hot encoded categorical features into a single dataframe

print("X_full_cat shape:", X_full_cat.shape)

# ----------------------------------------------------
# 5) numerical scaling on full training data
#    standardize numerical features so that they have zero mean and unit variance
# ----------------------------------------------------
scaler = StandardScaler()
X_full_num = scaler.fit_transform(X_full[numeric_features])

# convert back to dataframe with consistent index and prefixed column names
X_full_num_df = pd.DataFrame(
    X_full_num,
    index=X_full.index,
    columns=[f"num_{col}" for col in numeric_features]
)

print("X_full_num_df shape:", X_full_num_df.shape)


# ----------------------------------------------------
# build final training matrix by concatenating scaled numeric and encoded categorical features
#    will be used to train the final model
# ----------------------------------------------------
X_full_final = pd.concat(
    [X_full_num_df, X_full_cat],
    axis=1
)

print("X_full_final shape:", X_full_final.shape)


# ----------------------------------------------------
# train final random forest model on full training data using the chosen configuration
#    this is the model that will be used to generate predictions for the test set
# ----------------------------------------------------
rf_final = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **final_config
)

rf_final.fit(X_full_final, y_full)
print("Modelo final treinado com sucesso.")

# ----------------------------------------------------
# prepare test features using the exact same pipeline as for training
#    here we only apply transform operations using the states learned on the training data
# ----------------------------------------------------
test_df = pd.read_csv("../../project_data/test.csv")

# normalize string representation for the same set of categorical columns in the test data
cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]

for col in cols_to_normalize:
    if col in test_df.columns:
        test_df = column_string_transformer(
            test_df,
            column=col,
            remove_middle_spaces=True,
            allow_extra_chars=""
        )


# select the same base set of numeric and categorical columns as in training
X_test = test_df[numeric_features + categorical_features].copy()

# apply numerical imputers to the test data using training states only
X_test = transform_year_with_model_median(X_test, state=year_state)
X_test = transform_mileage_imputer(X_test, state=mileage_state)
X_test = transform_engine_size_imputer(X_test, state=engine_state)
X_test = transform_tax_imputer(X_test, state=tax_state)
X_test = transform_mpg_imputer(X_test, state=mpg_state)
X_test = transform_paint_quality_imputer(X_test, state=paint_state)
X_test = transform_previous_owners_imputer(X_test, state=owners_state)

# resolve categorical inconsistencies on test set using the same states learned from training
X_test, _, _ = transform_ambiguous_brands(X_test, brand_state)
X_test, _, _ = transform_invalid_models(X_test, model_state)
X_test, _, _ = transform_transmission_resolver(X_test, transm_state)
X_test, _, _ = transform_fueltype_resolver(X_test, fuel_state)

# apply the fitted target encoder and one hot encoder to the test data
X_test_high = te.transform(X_test[high_card_features])
X_test_low  = ohe.transform(X_test[low_card_features])

X_test_cat = pd.concat([X_test_high, X_test_low], axis=1)

# scale numerical test features using the scaler fitted on the full training data
X_test_num = scaler.transform(X_test[numeric_features])

X_test_num_df = pd.DataFrame(
    X_test_num,
    index=X_test.index,
    columns=[f"num_{col}" for col in numeric_features]
)

# build final test matrix by concatenating scaled numeric and encoded categorical test features
X_test_final = pd.concat(
    [X_test_num_df, X_test_cat],
    axis=1
)

print("X_test_final shape:", X_test_final.shape)

# ----------------------------------------------------
# generate test set predictions and build submission file
#    predictions are first inspected as floats and then optionally rounded to integers
# ----------------------------------------------------
y_test_pred = rf_final.predict(X_test_final)

print("Primeiras 10 previsões (float):", y_test_pred[:10])
print("Resumo das previsões (float):")
print(pd.Series(y_test_pred).describe())

# prices round the predictions to the nearest integer
y_test_pred_round = np.round(y_test_pred).astype(int)

# carid and predicted price
submission = pd.DataFrame({
    "carID": test_df["carID"].astype(int),
    "price": y_test_pred
})

submission_path = "rf_final_submission_config_1000_6_1_15_d.csv"
submission.to_csv(submission_path, index=False)

print("Ficheiro de submissão criado em:", submission_path)


Config final RF: {'n_estimators': 1000, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 15, 'bootstrap': True}
Features a manter: 100.0%
Correções aplicadas: 0
Brands ainda inválidas (serão tratadas pelo resolver robusto): 2
Após resolver Brand (train full): nº brands ainda inválidas: 0
Após resolver model (train full): nº models ainda inválidos: 1
Após resolver transmission (train full): valores distintos: ['AUTOMATIC', 'MANUAL', 'SEMIAUTO']
Após resolver fuelType (train full): valores distintos: ['DIESEL', 'ELECTRIC', 'HYBRID', 'OTHER', 'PETROL']
X_full_cat shape: (75973, 10)
X_full_num_df shape: (75973, 7)
X_full_final shape: (75973, 17)
Modelo final treinado com sucesso.
X_test_final shape: (32567, 17)
Primeiras 10 previsões (float): [ 8385.93548445 21923.69317011 14335.61472473 16777.98394774
 23704.29658898 10575.89043131 13355.48938839 14781.77002539
  4870.99690598 17846.67278988]
Resumo das previsões (float):
count     32567.000000
mean      1

In [1]:
# All of our preprocessing helper functions are in this notebook
%run 05_0_preproc_helpers.ipynb  

# this is the target column - the value we want to predict 
TARGET_COL = "price"  

# separate features and target variable from the full training datase
y = full_train_dataset[TARGET_COL].copy()
X = full_train_dataset.drop(columns=[TARGET_COL]).copy()

# at this point, this are all the features in use 
categorical_features = ['Brand', 'model', 'transmission', 'fuelType']              
numeric_features = ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'previousOwners']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Num features:", numeric_features)
print("Cat features:", categorical_features)

N_SPLITS = 8 # the number of folds for K-Fold cross-validation
RANDOM_STATE = 42 # seed to control randomness

X shape: (75973, 10)
y shape: (75973,)
Num features: ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'previousOwners']
Cat features: ['Brand', 'model', 'transmission', 'fuelType']


In [2]:
import os
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel

# ==============================================================================
# 0) SETTINGS
# ==============================================================================
RANDOM_STATE = 42

# hiperparâmetros 
final_config = {
    "n_estimators": 1000,
    "min_samples_split": 6,
    "min_samples_leaf": 1,
    "max_features": None,
    "max_depth": 15,
    "bootstrap": True,
}


FS_KEEP_RATIO = 0.65
RF_FS_PARAMS = {
    "n_estimators": 500,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "max_depth": None,
    "min_samples_split": 2,
    "min_samples_leaf": 2,
    "max_features": "sqrt",
    "bootstrap": True,
}

print(f"- Config final RF: {final_config}")
print(f"- FS_KEEP_RATIO: {FS_KEEP_RATIO}")

# ==============================================================================
# 1) LOAD TEST + IDS (igual ao HGB)
# ==============================================================================
try:
    test_df_raw = pd.read_csv("test.csv")
except:
    test_df_raw = pd.read_csv("../../project_data/test.csv")

ID_CANDIDATES = ["id", "carID", "carId", "car_id"]
ID_COL = next((c for c in ID_CANDIDATES if c in test_df_raw.columns), None)

if ID_COL is not None:
    test_ids = test_df_raw[ID_COL].copy()
    test_df = test_df_raw.drop(columns=[ID_COL]).copy()
else:
    test_ids = test_df_raw.index.copy()
    test_df = test_df_raw.copy()

# ==============================================================================
# 2) START DATA 
# ==============================================================================
X_full = X.copy()
y_full = y.copy()
y_full_log = np.log1p(y_full)

cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]
high_card_features = ["Brand", "model"]
low_card_features = [c for c in categorical_features if c not in high_card_features]


# ==============================================================================
# 4) STRING NORMALIZATION (TRAIN + TEST) 
# ==============================================================================
for col in cols_to_normalize:
    if col in X_full.columns:
        X_full[col] = fill_unknown(X_full[col])
        X_full = column_string_transformer(
            X_full, column=col, remove_middle_spaces=True, allow_extra_chars=""
        )
    if col in test_df.columns:
        test_df[col] = fill_unknown(test_df[col])
        test_df = column_string_transformer(
            test_df, column=col, remove_middle_spaces=True, allow_extra_chars=""
        )

# Restrict to expected columns (train schema)
expected_cols = [c for c in (numeric_features + categorical_features) if c in X_full.columns]
X_full = X_full[expected_cols].copy()
X_test = test_df.reindex(columns=expected_cols, fill_value=np.nan).copy()

# ==============================================================================
# 5) FIT + TRANSFORM ON FULL TRAIN (igual ao HGB)
# ==============================================================================
print("- fitting transforms on full train")

year_state = fit_year_median(X_full, year_col="year", model_col="model")
X_full = transform_year_with_model_median(X_full, state=year_state)

engine_state = fit_engine_size_imputer(X_full, engine_col="engineSize")
X_full = transform_engine_size_imputer(X_full, state=engine_state)

X_full = transform_tax_custom_rules(X_full, "tax", "year", "fuelType", "engineSize")

mpg_state = fit_mpg_imputer(X_full, mpg_col="mpg", do_abs=True)
X_full = transform_mpg_imputer(X_full, state=mpg_state)

# IMPORTANT: owners imputer must happen BEFORE dropping previousOwners
owners_state = fit_previous_owners_imputer(X_full, "previousOwners", "year", "mileage")
X_full = transform_previous_owners_imputer(X_full, state=owners_state)

brand_state = fit_ambiguous_brand_resolver(X_full, valid_brands)
X_full, _, _ = transform_ambiguous_brands(X_full, brand_state)

model_state = fit_invalid_model_resolver(X_full, valid_models_by_brand)
X_full, _, _ = transform_invalid_models(X_full, model_state)

transm_state = fit_transmission_resolver(X_full, valid_transmissions)
X_full, _, _ = transform_transmission_resolver(X_full, transm_state)

fuel_state = fit_fueltype_resolver(X_full, valid_fueltypes)
X_full, _, _ = transform_fueltype_resolver(X_full, fuel_state)

# ==============================================================================
# 6) FEATURE ENGINEERING (FULL TRAIN) 
# ==============================================================================
print("- creating engineered features on full train")

X_full = add_owners_flagged(X_full, owners_col="previousOwners", new_col="owners_flagged", drop_original=True)
X_full = create_age_and_drop_year(X_full, year_col="year", base_year=2020)
X_full = add_mileage_features(X_full, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)
X_full = add_engine_bins(X_full, engine_col="engineSize", new_col="engine_bin")

# engine_bin as low-card categorical
X_full["engine_bin"] = X_full["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")
low_card_curr = low_card_features + ["engine_bin"]

# ==============================================================================
# 7) ENCODING (FIT ON FULL TRAIN)
# ==============================================================================
print("- fitting encoders on full train")

te = MyTargetEncoder(smoothing=5)
te.fit(X_full[high_card_features], y_full_log)
X_full_high = te.transform(X_full[high_card_features])

ohe = MyOneHotEncoder()
ohe.fit(X_full[low_card_curr])
X_full_low = ohe.transform(X_full[low_card_curr])

X_full_cat = pd.concat([X_full_high, X_full_low], axis=1)

drop_for_numeric = set(high_card_features + low_card_curr)
numeric_features_curr = [c for c in X_full.columns if c not in drop_for_numeric]

X_full_final = pd.concat([X_full[numeric_features_curr], X_full_cat], axis=1)
print(f"- train matrix shape: {X_full_final.shape}")

# ==============================================================================
# 8) FEATURE SELECTION (FIT ON FULL TRAIN) 
# ==============================================================================
print("- fitting feature selector (RF)")

n_feats = X_full_final.shape[1]
k = int(np.ceil(FS_KEEP_RATIO * n_feats))
k = max(1, min(k, n_feats))

rf_fs = RandomForestRegressor(**RF_FS_PARAMS)
rf_fs.fit(X_full_final, y_full_log)

selector = SelectFromModel(
    estimator=rf_fs,
    threshold=-np.inf,
    max_features=k,
    prefit=True
)

selected_cols = X_full_final.columns[selector.get_support()]
X_full_sel = X_full_final[selected_cols]
print(f"- selected features: {len(selected_cols)}/{n_feats}")

# ==============================================================================
# 9) TRAIN FINAL RF (LOG TARGET)  
# ==============================================================================
print("- training final RF (log target)")

rf_final = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **final_config
)
rf_final.fit(X_full_sel, y_full_log)
print("- model trained")

# ==============================================================================
# 10) TRANSFORM TEST 
# ==============================================================================
print("- transforming test set")

X_test = transform_year_with_model_median(X_test, state=year_state)
X_test = transform_engine_size_imputer(X_test, state=engine_state)
X_test = transform_tax_custom_rules(X_test, "tax", "year", "fuelType", "engineSize")
X_test = transform_mpg_imputer(X_test, state=mpg_state)

X_test = transform_previous_owners_imputer(X_test, state=owners_state)

X_test, _, _ = transform_ambiguous_brands(X_test, brand_state)
X_test, _, _ = transform_invalid_models(X_test, model_state)
X_test, _, _ = transform_transmission_resolver(X_test, transm_state)
X_test, _, _ = transform_fueltype_resolver(X_test, fuel_state)

X_test = add_owners_flagged(X_test, owners_col="previousOwners", new_col="owners_flagged", drop_original=True)
X_test = create_age_and_drop_year(X_test, year_col="year", base_year=2020)
X_test = add_mileage_features(X_test, mileage_col="mileage", age_col="age", drop_original=True, drop_ratio=True)
X_test = add_engine_bins(X_test, engine_col="engineSize", new_col="engine_bin")

X_test["engine_bin"] = X_test["engine_bin"].astype("Int64").astype("string").fillna("UNKNOWN")

# encoding transform-only
X_test_high = te.transform(X_test[high_card_features])
X_test_low  = ohe.transform(X_test[low_card_curr])
X_test_cat  = pd.concat([X_test_high, X_test_low], axis=1)

drop_for_numeric = set(high_card_features + low_card_curr)
numeric_features_curr_test = [c for c in X_test.columns if c not in drop_for_numeric]

X_test_final = pd.concat([X_test[numeric_features_curr_test], X_test_cat], axis=1)

# align to train columns before selecting
X_test_final = X_test_final.reindex(columns=X_full_final.columns, fill_value=0)

# apply same selected columns
X_test_sel = X_test_final.reindex(columns=selected_cols, fill_value=0)

print(f"- test matrix shape: {X_test_sel.shape}")

# ==============================================================================
# 11) PREDICT + SUBMISSION
# ==============================================================================
pred_log = rf_final.predict(X_test_sel)
pred_final = np.expm1(pred_log)
pred_final = np.maximum(pred_final, 0)

submission = pd.DataFrame({
    (ID_COL if ID_COL is not None else "id"): test_ids,
    "price": pred_final
})

submission_filename = "submission_rf_log_target_fs065.csv"
submission.to_csv(submission_filename, index=False)

print(f"- saved: {submission_filename}")
print(submission.head())


- Config final RF: {'n_estimators': 1000, 'min_samples_split': 6, 'min_samples_leaf': 1, 'max_features': None, 'max_depth': 15, 'bootstrap': True}
- FS_KEEP_RATIO: 0.65
- fitting transforms on full train
- creating engineered features on full train
- fitting encoders on full train
- train matrix shape: (75973, 25)
- fitting feature selector (RF)
- selected features: 17/25
- training final RF (log target)
- model trained
- transforming test set
- test matrix shape: (32567, 17)
- saved: submission_rf_log_target_fs065.csv
    carID         price
0   89856  11583.228126
1  106581  24982.627453
2   80886  14166.922095
3  100174  16619.799909
4   81376  23085.463528
